<a href="https://colab.research.google.com/github/Harini-Sri-Vedha-Alvar/Fake-Job-Post-Prediction-ML-Project-/blob/main/FAKE_JOB_DETECTION_WEB_APP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# Install the packages required for the fake job detection web application.
!pip -q install flask pypdf pillow pytesseract joblib scikit-learn pandas numpy
!apt-get -qq update
!apt-get -qq install -y tesseract-ocr

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [7]:
# Connect Google Drive so the web application can access the trained ML pipeline.
from google.colab import drive

drive.mount("/content/drive")

print("Google Drive connected successfully.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive connected successfully.


In [8]:
# Load the trained fake job detection pipeline and verify that its components are available.
import os
import joblib

MODEL_PATH = "/content/drive/MyDrive/fake_job_detection_model/fake_job_detection_pipeline.joblib"

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        f"Model not found at: {MODEL_PATH}\n"
        "Please run the final model-saving cell in Notebook 1 first."
    )

pipeline = joblib.load(MODEL_PATH)

model = pipeline["model"]
tfidf = pipeline["tfidf"]
threshold = pipeline["threshold"]

print("✅ Model loaded successfully!")
print("Model:", type(model).__name__)
print("Vectorizer:", type(tfidf).__name__)
print("Decision threshold:", threshold)

✅ Model loaded successfully!
Model: LogisticRegression
Vectorizer: TfidfVectorizer
Decision threshold: 0.6599999999999997


In [9]:
# Test the loaded ML pipeline with a sample job posting.
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

sample_job = """
We are hiring a Data Analyst with experience in Python and SQL.
The successful candidate will analyze business data and create reports.
Please apply through the official company recruitment process.
"""

cleaned_text = clean_text(sample_job)
features = tfidf.transform([cleaned_text])

fraud_probability = model.predict_proba(features)[0, 1]
prediction = fraud_probability >= threshold

print("Fraud probability:", round(fraud_probability * 100, 2), "%")
print("Threshold:", round(threshold, 3))
print("Prediction:", "Potentially Fraudulent" if prediction else "Likely Legitimate")

Fraud probability: 13.86 %
Threshold: 0.66
Prediction: Likely Legitimate


In [10]:
# Create the Flask web application structure and initialize the prediction endpoint.
from flask import Flask, request, jsonify
import threading
import os

app = Flask(__name__)

@app.route("/health", methods=["GET"])
def health():
    return jsonify({
        "status": "online",
        "model": type(model).__name__,
        "message": "Fake Job Detection AI is ready."
    })

@app.route("/predict", methods=["POST"])
def predict():
    data = request.get_json(silent=True) or {}
    job_text = data.get("text", "").strip()

    if not job_text:
        return jsonify({
            "error": "Please provide job posting text."
        }), 400

    cleaned_text = clean_text(job_text)
    features = tfidf.transform([cleaned_text])
    probability = float(model.predict_proba(features)[0, 1])

    is_fraudulent = probability >= threshold

    return jsonify({
        "prediction": (
            "Potentially Fraudulent"
            if is_fraudulent
            else "Likely Legitimate"
        ),
        "fraud_probability": round(probability * 100, 2),
        "threshold": round(threshold * 100, 2)
    })

print("✅ Flask application created successfully.")
print("Available endpoints:")
print("  /health")
print("  /predict")

✅ Flask application created successfully.
Available endpoints:
  /health
  /predict


In [11]:
# Create the folders and files needed for the professional FakeGuard AI web interface.
from pathlib import Path

BASE_DIR = Path("/content/fakeguard_app")
TEMPLATES_DIR = BASE_DIR / "templates"
STATIC_DIR = BASE_DIR / "static"

TEMPLATES_DIR.mkdir(parents=True, exist_ok=True)
STATIC_DIR.mkdir(parents=True, exist_ok=True)

print("✅ FakeGuard AI application folders created.")
print("📁", BASE_DIR)

✅ FakeGuard AI application folders created.
📁 /content/fakeguard_app


In [12]:
# Create the professional HTML interface for FakeGuard AI.
html_content = r"""
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>FakeGuard AI | Job Risk Analyzer</title>
    <link rel="stylesheet" href="/static/style.css">
    <link rel="preconnect" href="https://fonts.googleapis.com">
    <link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
    <link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800&display=swap" rel="stylesheet">
</head>

<body>

    <div class="background-glow glow-one"></div>
    <div class="background-glow glow-two"></div>

    <header class="navbar">
        <div class="brand">
            <div class="brand-icon">🛡️</div>
            <div>
                <h1>FakeGuard <span>AI</span></h1>
                <p>Job Posting Risk Analyzer</p>
            </div>
        </div>

        <div class="status">
            <span class="status-dot"></span>
            AI MODEL ONLINE
        </div>
    </header>

    <main class="container">

        <section class="hero">
            <div class="badge">🤖 NLP + MACHINE LEARNING</div>

            <h2>
                Detect suspicious<br>
                <span>job postings with AI.</span>
            </h2>

            <p>
                Analyze a job posting using our trained NLP machine-learning
                model and receive a fraud-risk assessment in seconds.
            </p>
        </section>

        <section class="analyzer-card">

            <div class="section-heading">
                <div>
                    <h3>Analyze Job Posting</h3>
                    <p>Paste the job description below to begin your analysis.</p>
                </div>
                <div class="secure-badge">🔒 PRIVATE ANALYSIS</div>
            </div>

            <textarea
                id="jobText"
                placeholder="Paste the complete job posting here...

Example:
We are looking for a Data Analyst with experience in Python and SQL...
"
            ></textarea>

            <div class="input-info">
                <span id="characterCount">0 characters</span>
                <span>Text is processed by the trained NLP model</span>
            </div>

            <div class="action-row">
                <button id="analyzeButton" onclick="analyzeJob()">
                    <span>🔍</span>
                    ANALYZE JOB POSTING
                </button>

                <button class="clear-button" onclick="clearAnalysis()">
                    Clear
                </button>
            </div>

            <div class="upload-divider">
                <span>OR USE A FILE</span>
            </div>

            <div class="upload-grid">

                <div class="upload-card">
                    <div class="upload-icon">📄</div>
                    <h4>Upload PDF</h4>
                    <p>Analyze a job posting PDF</p>
                    <button class="upload-button" onclick="document.getElementById('pdfInput').click()">
                        Choose PDF
                    </button>
                    <input id="pdfInput" type="file" accept=".pdf" hidden>
                </div>

                <div class="upload-card">
                    <div class="upload-icon">🖼️</div>
                    <h4>Upload Image</h4>
                    <p>Analyze a screenshot</p>
                    <button class="upload-button" onclick="document.getElementById('imageInput').click()">
                        Choose Image
                    </button>
                    <input id="imageInput" type="file" accept="image/*" hidden>
                </div>

            </div>

        </section>

        <section id="loadingSection" class="loading-section hidden">
            <div class="loader"></div>
            <h3>Analyzing Job Posting...</h3>
            <p>Our NLP model is examining the provided content.</p>
        </section>

        <section id="resultSection" class="result-card hidden">

            <div class="result-header">
                <div>
                    <p class="result-label">AI ANALYSIS RESULT</p>
                    <h3 id="predictionText">Analysis Complete</h3>
                </div>

                <div id="resultIcon" class="result-icon">
                    🔍
                </div>
            </div>

            <div class="risk-display">

                <div class="risk-number">
                    <span id="probabilityText">0%</span>
                    <small>fraud risk</small>
                </div>

                <div class="risk-meter">
                    <div id="riskBar" class="risk-bar"></div>
                </div>

            </div>

            <div class="result-details">

                <div class="detail-box">
                    <span>Prediction</span>
                    <strong id="predictionDetail">—</strong>
                </div>

                <div class="detail-box">
                    <span>Fraud Probability</span>
                    <strong id="probabilityDetail">—</strong>
                </div>

                <div class="detail-box">
                    <span>Model</span>
                    <strong>Logistic Regression</strong>
                </div>

            </div>

            <div class="disclaimer">
                ⚠️ <strong>Important:</strong>
                This prediction is a machine-learning-based risk signal.
                It is not definitive proof that a job posting is fraudulent.
            </div>

        </section>

        <section class="how-section">

            <div class="section-title">
                <span>HOW IT WORKS</span>
                <h3>From job posting to AI insight</h3>
            </div>

            <div class="steps">

                <div class="step">
                    <div class="step-number">01</div>
                    <h4>Input</h4>
                    <p>Provide job posting text, PDF, or image.</p>
                </div>

                <div class="step">
                    <div class="step-number">02</div>
                    <h4>NLP</h4>
                    <p>Clean and transform the job content using TF-IDF.</p>
                </div>

                <div class="step">
                    <div class="step-number">03</div>
                    <h4>Prediction</h4>
                    <p>The trained ML model calculates fraud probability.</p>
                </div>

                <div class="step">
                    <div class="step-number">04</div>
                    <h4>Insight</h4>
                    <p>Receive an easy-to-understand risk assessment.</p>
                </div>

            </div>

        </section>

    </main>

    <footer>
        <div>
            <strong>🛡️ FakeGuard AI</strong>
            <span> | NLP-Based Fake Job Detection</span>
        </div>
        <p>Built with Python • Flask • Scikit-learn • TF-IDF</p>
    </footer>

    <script src="/static/script.js"></script>

</body>
</html>
"""

(TEMPLATES_DIR / "index.html").write_text(html_content, encoding="utf-8")

print("✅ Professional HTML interface created.")

✅ Professional HTML interface created.


In [13]:
# Create the responsive professional CSS styling for FakeGuard AI.
css_content = r"""
* {
    margin: 0;
    padding: 0;
    box-sizing: border-box;
}

:root {
    --navy: #07111f;
    --navy-light: #0d1b2e;
    --blue: #3b82f6;
    --cyan: #06b6d4;
    --purple: #8b5cf6;
    --green: #22c55e;
    --red: #ef4444;
    --white: #f8fafc;
    --muted: #94a3b8;
    --border: rgba(148, 163, 184, 0.16);
}

body {
    font-family: "Inter", sans-serif;
    background:
        radial-gradient(circle at 15% 10%, rgba(59, 130, 246, 0.18), transparent 28%),
        radial-gradient(circle at 85% 20%, rgba(139, 92, 246, 0.16), transparent 28%),
        var(--navy);
    color: var(--white);
    min-height: 100vh;
    overflow-x: hidden;
}

.background-glow {
    position: fixed;
    width: 350px;
    height: 350px;
    border-radius: 50%;
    filter: blur(100px);
    pointer-events: none;
    z-index: -1;
}

.glow-one {
    background: rgba(6, 182, 212, 0.08);
    top: 40%;
    left: -180px;
}

.glow-two {
    background: rgba(139, 92, 246, 0.08);
    bottom: 5%;
    right: -180px;
}

.navbar {
    max-width: 1200px;
    margin: auto;
    padding: 28px 24px;
    display: flex;
    justify-content: space-between;
    align-items: center;
}

.brand {
    display: flex;
    align-items: center;
    gap: 14px;
}

.brand-icon {
    width: 48px;
    height: 48px;
    border-radius: 14px;
    display: flex;
    align-items: center;
    justify-content: center;
    font-size: 24px;
    background: linear-gradient(135deg, var(--blue), var(--purple));
    box-shadow: 0 10px 30px rgba(59, 130, 246, 0.25);
}

.brand h1 {
    font-size: 20px;
    font-weight: 800;
}

.brand h1 span {
    color: #60a5fa;
}

.brand p {
    color: var(--muted);
    font-size: 11px;
    margin-top: 2px;
}

.status {
    display: flex;
    align-items: center;
    gap: 8px;
    padding: 9px 14px;
    border: 1px solid rgba(34, 197, 94, 0.2);
    border-radius: 999px;
    background: rgba(34, 197, 94, 0.06);
    color: #86efac;
    font-size: 11px;
    font-weight: 700;
}

.status-dot {
    width: 7px;
    height: 7px;
    border-radius: 50%;
    background: var(--green);
    box-shadow: 0 0 12px var(--green);
}

.container {
    max-width: 1050px;
    margin: auto;
    padding: 45px 24px 80px;
}

.hero {
    text-align: center;
    max-width: 760px;
    margin: auto;
}

.badge {
    display: inline-block;
    padding: 8px 15px;
    border-radius: 999px;
    background: rgba(59, 130, 246, 0.1);
    border: 1px solid rgba(59, 130, 246, 0.22);
    color: #93c5fd;
    font-size: 11px;
    font-weight: 700;
    letter-spacing: 1px;
    margin-bottom: 22px;
}

.hero h2 {
    font-size: clamp(40px, 6vw, 68px);
    line-height: 1.05;
    letter-spacing: -2px;
    font-weight: 800;
}

.hero h2 span {
    background: linear-gradient(90deg, #60a5fa, #22d3ee, #a78bfa);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
}

.hero p {
    color: var(--muted);
    max-width: 650px;
    margin: 24px auto 0;
    line-height: 1.7;
    font-size: 15px;
}

.analyzer-card,
.result-card {
    margin-top: 45px;
    padding: 30px;
    border: 1px solid var(--border);
    border-radius: 24px;
    background: rgba(13, 27, 46, 0.78);
    backdrop-filter: blur(20px);
    box-shadow: 0 30px 80px rgba(0, 0, 0, 0.25);
}

.section-heading {
    display: flex;
    justify-content: space-between;
    align-items: center;
    gap: 20px;
    margin-bottom: 22px;
}

.section-heading h3 {
    font-size: 22px;
}

.section-heading p {
    color: var(--muted);
    margin-top: 5px;
    font-size: 13px;
}

.secure-badge {
    color: #93c5fd;
    font-size: 10px;
    font-weight: 700;
    white-space: nowrap;
}

textarea {
    width: 100%;
    min-height: 270px;
    resize: vertical;
    border: 1px solid var(--border);
    border-radius: 16px;
    background: rgba(2, 6, 23, 0.7);
    color: var(--white);
    padding: 20px;
    font-family: inherit;
    font-size: 14px;
    line-height: 1.7;
    outline: none;
    transition: 0.25s;
}

textarea:focus {
    border-color: rgba(59, 130, 246, 0.65);
    box-shadow: 0 0 0 4px rgba(59, 130, 246, 0.08);
}

textarea::placeholder {
    color: #64748b;
}

.input-info {
    display: flex;
    justify-content: space-between;
    color: #64748b;
    font-size: 11px;
    margin: 10px 3px 18px;
}

.action-row {
    display: flex;
    gap: 12px;
}

#analyzeButton {
    flex: 1;
    border: none;
    border-radius: 13px;
    padding: 16px;
    font-weight: 800;
    color: white;
    cursor: pointer;
    background: linear-gradient(90deg, #2563eb, #7c3aed);
    box-shadow: 0 12px 30px rgba(59, 130, 246, 0.22);
    transition: 0.25s;
}

#analyzeButton:hover {
    transform: translateY(-2px);
    box-shadow: 0 18px 35px rgba(59, 130, 246, 0.3);
}

.clear-button {
    padding: 0 22px;
    border: 1px solid var(--border);
    border-radius: 13px;
    background: transparent;
    color: var(--muted);
    cursor: pointer;
}

.upload-divider {
    display: flex;
    align-items: center;
    gap: 15px;
    margin: 28px 0;
    color: #64748b;
    font-size: 10px;
    font-weight: 700;
}

.upload-divider::before,
.upload-divider::after {
    content: "";
    height: 1px;
    flex: 1;
    background: var(--border);
}

.upload-grid {
    display: grid;
    grid-template-columns: 1fr 1fr;
    gap: 16px;
}

.upload-card {
    text-align: center;
    padding: 25px;
    border: 1px dashed rgba(148, 163, 184, 0.25);
    border-radius: 17px;
    background: rgba(2, 6, 23, 0.25);
    transition: 0.25s;
}

.upload-card:hover {
    border-color: rgba(59, 130, 246, 0.5);
    transform: translateY(-2px);
}

.upload-icon {
    font-size: 30px;
    margin-bottom: 10px;
}

.upload-card h4 {
    font-size: 15px;
}

.upload-card p {
    color: var(--muted);
    font-size: 11px;
    margin: 6px 0 15px;
}

.upload-button {
    padding: 9px 17px;
    border: 1px solid rgba(59, 130, 246, 0.3);
    border-radius: 9px;
    color: #93c5fd;
    background: rgba(59, 130, 246, 0.08);
    cursor: pointer;
}

.loading-section {
    text-align: center;
    margin-top: 30px;
    padding: 40px;
}

.loader {
    width: 42px;
    height: 42px;
    margin: auto auto 15px;
    border: 4px solid rgba(255,255,255,0.1);
    border-top-color: #60a5fa;
    border-right-color: #a78bfa;
    border-radius: 50%;
    animation: spin 0.8s linear infinite;
}

@keyframes spin {
    to {
        transform: rotate(360deg);
    }
}

.hidden {
    display: none !important;
}

.result-card {
    border-color: rgba(59, 130, 246, 0.22);
}

.result-header {
    display: flex;
    justify-content: space-between;
    align-items: center;
}

.result-label {
    color: #60a5fa;
    font-size: 10px;
    font-weight: 800;
    letter-spacing: 1.5px;
}

.result-header h3 {
    margin-top: 8px;
    font-size: 27px;
}

.result-icon {
    width: 60px;
    height: 60px;
    border-radius: 18px;
    display: flex;
    align-items: center;
    justify-content: center;
    font-size: 27px;
    background: rgba(59, 130, 246, 0.1);
}

.risk-display {
    margin: 30px 0;
}

.risk-number {
    display: flex;
    align-items: baseline;
    gap: 10px;
}

.risk-number span {
    font-size: 54px;
    font-weight: 800;
}

.risk-number small {
    color: var(--muted);
}

.risk-meter {
    height: 13px;
    border-radius: 999px;
    background: rgba(148, 163, 184, 0.12);
    overflow: hidden;
    margin-top: 15px;
}

.risk-bar {
    height: 100%;
    width: 0%;
    border-radius: inherit;
    background: linear-gradient(90deg, #22c55e, #facc15, #ef4444);
    transition: width 1s ease;
}

.result-details {
    display: grid;
    grid-template-columns: repeat(3, 1fr);
    gap: 12px;
}

.detail-box {
    padding: 18px;
    border-radius: 14px;
    background: rgba(2, 6, 23, 0.4);
    border: 1px solid var(--border);
}

.detail-box span {
    display: block;
    color: #64748b;
    font-size: 10px;
    text-transform: uppercase;
    margin-bottom: 7px;
}

.detail-box strong {
    font-size: 13px;
}

.disclaimer {
    margin-top: 20px;
    padding: 15px;
    border-radius: 12px;
    background: rgba(245, 158, 11, 0.06);
    border: 1px solid rgba(245, 158, 11, 0.15);
    color: #cbd5e1;
    font-size: 11px;
    line-height: 1.6;
}

.how-section {
    margin-top: 80px;
}

.section-title {
    text-align: center;
    margin-bottom: 30px;
}

.section-title span {
    color: #60a5fa;
    font-size: 10px;
    font-weight: 800;
    letter-spacing: 2px;
}

.section-title h3 {
    margin-top: 8px;
    font-size: 28px;
}

.steps {
    display: grid;
    grid-template-columns: repeat(4, 1fr);
    gap: 15px;
}

.step {
    padding: 22px;
    border: 1px solid var(--border);
    border-radius: 17px;
    background: rgba(13, 27, 46, 0.55);
}

.step-number {
    color: #60a5fa;
    font-size: 12px;
    font-weight: 800;
    margin-bottom: 18px;
}

.step h4 {
    margin-bottom: 8px;
}

.step p {
    color: var(--muted);
    font-size: 11px;
    line-height: 1.6;
}

footer {
    max-width: 1050px;
    margin: auto;
    padding: 25px 24px 45px;
    border-top: 1px solid var(--border);
    display: flex;
    justify-content: space-between;
    color: #64748b;
    font-size: 11px;
}

footer strong {
    color: #cbd5e1;
}

@media (max-width: 750px) {

    .navbar {
        padding-top: 20px;
    }

    .status {
        display: none;
    }

    .container {
        padding-top: 30px;
    }

    .hero h2 {
        font-size: 42px;
    }

    .section-heading {
        align-items: flex-start;
        flex-direction: column;
    }

    .upload-grid,
    .result-details,
    .steps {
        grid-template-columns: 1fr;
    }

    .action-row {
        flex-direction: column;
    }

    .clear-button {
        padding: 14px;
    }

    footer {
        flex-direction: column;
        gap: 8px;
    }
}
"""

(STATIC_DIR / "style.css").write_text(css_content, encoding="utf-8")

print("🎨 Professional FakeGuard AI styling created.")

🎨 Professional FakeGuard AI styling created.


In [14]:
# Create the JavaScript logic for job analysis, loading states, results, and character counting.
js_content = r"""
const jobText = document.getElementById("jobText");
const characterCount = document.getElementById("characterCount");

jobText.addEventListener("input", function () {
    characterCount.textContent = `${jobText.value.length} characters`;
});

async function analyzeJob() {

    const text = jobText.value.trim();

    if (!text) {
        alert("Please paste a job posting before analyzing.");
        return;
    }

    const loadingSection = document.getElementById("loadingSection");
    const resultSection = document.getElementById("resultSection");
    const analyzeButton = document.getElementById("analyzeButton");

    resultSection.classList.add("hidden");
    loadingSection.classList.remove("hidden");

    analyzeButton.disabled = true;
    analyzeButton.innerHTML = "⏳ ANALYZING...";

    try {

        const response = await fetch("/predict", {
            method: "POST",
            headers: {
                "Content-Type": "application/json"
            },
            body: JSON.stringify({
                text: text
            })
        });

        const data = await response.json();

        if (!response.ok) {
            throw new Error(data.error || "Analysis failed.");
        }

        showResult(data);

    } catch (error) {

        alert(error.message);

    } finally {

        loadingSection.classList.add("hidden");
        analyzeButton.disabled = false;
        analyzeButton.innerHTML = "<span>🔍</span> ANALYZE JOB POSTING";
    }
}

function showResult(data) {

    const resultSection = document.getElementById("resultSection");
    const predictionText = document.getElementById("predictionText");
    const predictionDetail = document.getElementById("predictionDetail");
    const probabilityText = document.getElementById("probabilityText");
    const probabilityDetail = document.getElementById("probabilityDetail");
    const riskBar = document.getElementById("riskBar");
    const resultIcon = document.getElementById("resultIcon");

    const probability = Number(data.fraud_probability);

    predictionText.textContent = data.prediction;
    predictionDetail.textContent = data.prediction;
    probabilityText.textContent = `${probability}%`;
    probabilityDetail.textContent = `${probability}%`;

    riskBar.style.width = `${probability}%`;

    if (data.prediction === "Potentially Fraudulent") {

        resultIcon.textContent = "⚠️";
        predictionText.style.color = "#f87171";

    } else {

        resultIcon.textContent = "✅";
        predictionText.style.color = "#4ade80";
    }

    resultSection.classList.remove("hidden");

    resultSection.scrollIntoView({
        behavior: "smooth",
        block: "center"
    });
}

function clearAnalysis() {

    jobText.value = "";
    characterCount.textContent = "0 characters";

    document.getElementById("resultSection").classList.add("hidden");
    document.getElementById("loadingSection").classList.add("hidden");
}
"""

(STATIC_DIR / "script.js").write_text(js_content, encoding="utf-8")

print("⚡ JavaScript interface logic created.")

⚡ JavaScript interface logic created.


In [15]:
# Connect the Flask backend to the new FakeGuard AI frontend and prepare the application server.
from flask import render_template
from google.colab import output
import threading
import time

@app.route("/", methods=["GET"])
def home():
    return render_template("index.html")

app.template_folder = str(TEMPLATES_DIR)
app.static_folder = str(STATIC_DIR)
app.static_url_path = "/static"

def run_app():
    app.run(
        host="0.0.0.0",
        port=5000,
        debug=False,
        use_reloader=False
    )

server_thread = threading.Thread(target=run_app, daemon=True)
server_thread.start()

time.sleep(2)

print("🚀 FakeGuard AI web application is running.")
print("🌐 Opening the application...")

output.serve_kernel_port_as_window(5000)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


🚀 FakeGuard AI web application is running.
🌐 Opening the application...
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [16]:
# Add PDF text extraction functionality to the FakeGuard AI backend.
from pypdf import PdfReader
from werkzeug.utils import secure_filename

def extract_pdf_text(file):
    reader = PdfReader(file)
    pages = []

    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            pages.append(page_text)

    return "\n".join(pages).strip()

print("✅ PDF extraction function ready.")

✅ PDF extraction function ready.


In [17]:
# Add OCR functionality for extracting job-posting text from uploaded images.
from PIL import Image
import pytesseract

def extract_image_text(file):
    image = Image.open(file)
    text = pytesseract.image_to_string(image)
    return text.strip()

print("✅ Image OCR function ready.")

✅ Image OCR function ready.


In [18]:
# Create a reusable function that sends extracted job-posting text through the trained ML pipeline.
def analyze_job_text(job_text):
    if not job_text or not job_text.strip():
        return {
            "error": "No readable job-posting text was found."
        }

    cleaned_text = clean_text(job_text)
    features = tfidf.transform([cleaned_text])
    probability = float(model.predict_proba(features)[0, 1])

    return {
        "prediction": (
            "Potentially Fraudulent"
            if probability >= threshold
            else "Likely Legitimate"
        ),
        "fraud_probability": round(probability * 100, 2),
        "threshold": round(threshold * 100, 2)
    }

print("✅ Unified job analysis function ready.")

✅ Unified job analysis function ready.


In [19]:
# Add Flask endpoints for PDF and image job-posting analysis.
@app.route("/analyze-file", methods=["POST"])
def analyze_file():

    if "file" not in request.files:
        return jsonify({
            "error": "No file was uploaded."
        }), 400

    uploaded_file = request.files["file"]

    if not uploaded_file.filename:
        return jsonify({
            "error": "Please select a file."
        }), 400

    filename = secure_filename(uploaded_file.filename)
    extension = filename.lower().split(".")[-1]

    try:

        if extension == "pdf":
            extracted_text = extract_pdf_text(uploaded_file)

        elif extension in ["png", "jpg", "jpeg", "webp"]:
            extracted_text = extract_image_text(uploaded_file)

        else:
            return jsonify({
                "error": "Supported formats are PDF, PNG, JPG, JPEG, and WEBP."
            }), 400

        if not extracted_text:
            return jsonify({
                "error": "Could not extract readable text from this file."
            }), 400

        result = analyze_job_text(extracted_text)
        result["extracted_characters"] = len(extracted_text)

        return jsonify(result)

    except Exception as error:

        return jsonify({
            "error": f"File analysis failed: {str(error)}"
        }), 500

print("✅ PDF and image analysis endpoint added.")

✅ PDF and image analysis endpoint added.


In [20]:
# Update the JavaScript so PDF and image uploads can be analyzed by the ML backend.
js_content = r"""
const jobText = document.getElementById("jobText");
const characterCount = document.getElementById("characterCount");

jobText.addEventListener("input", function () {
    characterCount.textContent = `${jobText.value.length} characters`;
});

async function analyzeJob() {

    const text = jobText.value.trim();

    if (!text) {
        alert("Please paste a job posting before analyzing.");
        return;
    }

    showLoading();

    try {

        const response = await fetch("/predict", {
            method: "POST",
            headers: {
                "Content-Type": "application/json"
            },
            body: JSON.stringify({
                text: text
            })
        });

        const data = await response.json();

        if (!response.ok) {
            throw new Error(data.error || "Analysis failed.");
        }

        showResult(data);

    } catch (error) {

        alert(error.message);

    } finally {

        hideLoading();
    }
}

async function analyzeFile(file) {

    if (!file) {
        return;
    }

    showLoading();

    const formData = new FormData();
    formData.append("file", file);

    try {

        const response = await fetch("/analyze-file", {
            method: "POST",
            body: formData
        });

        const data = await response.json();

        if (!response.ok) {
            throw new Error(data.error || "File analysis failed.");
        }

        showResult(data);

    } catch (error) {

        alert(error.message);

    } finally {

        hideLoading();
    }
}

function showLoading() {

    document.getElementById("resultSection").classList.add("hidden");
    document.getElementById("loadingSection").classList.remove("hidden");

    const button = document.getElementById("analyzeButton");

    button.disabled = true;
    button.innerHTML = "⏳ ANALYZING...";
}

function hideLoading() {

    document.getElementById("loadingSection").classList.add("hidden");

    const button = document.getElementById("analyzeButton");

    button.disabled = false;
    button.innerHTML = "<span>🔍</span> ANALYZE JOB POSTING";
}

function showResult(data) {

    const resultSection = document.getElementById("resultSection");
    const predictionText = document.getElementById("predictionText");
    const predictionDetail = document.getElementById("predictionDetail");
    const probabilityText = document.getElementById("probabilityText");
    const probabilityDetail = document.getElementById("probabilityDetail");
    const riskBar = document.getElementById("riskBar");
    const resultIcon = document.getElementById("resultIcon");

    const probability = Number(data.fraud_probability);

    predictionText.textContent = data.prediction;
    predictionDetail.textContent = data.prediction;
    probabilityText.textContent = `${probability}%`;
    probabilityDetail.textContent = `${probability}%`;

    riskBar.style.width = `${probability}%`;

    if (data.prediction === "Potentially Fraudulent") {

        resultIcon.textContent = "⚠️";
        predictionText.style.color = "#f87171";

    } else {

        resultIcon.textContent = "✅";
        predictionText.style.color = "#4ade80";
    }

    resultSection.classList.remove("hidden");

    resultSection.scrollIntoView({
        behavior: "smooth",
        block: "center"
    });
}

function clearAnalysis() {

    jobText.value = "";
    characterCount.textContent = "0 characters";

    document.getElementById("resultSection").classList.add("hidden");
    document.getElementById("loadingSection").classList.add("hidden");

    document.getElementById("pdfInput").value = "";
    document.getElementById("imageInput").value = "";
}

document.getElementById("pdfInput").addEventListener("change", function () {
    analyzeFile(this.files[0]);
});

document.getElementById("imageInput").addEventListener("change", function () {
    analyzeFile(this.files[0]);
});
"""

(STATIC_DIR / "script.js").write_text(js_content, encoding="utf-8")

print("✅ PDF and image upload controls connected.")

✅ PDF and image upload controls connected.


In [21]:
# Install the packages required to fetch and extract text from public job-posting URLs.
!pip -q install requests beautifulsoup4
print("✅ URL analysis packages installed.")

✅ URL analysis packages installed.


In [22]:
# Create a function that downloads a public web page and extracts readable job-posting text.
import requests
from bs4 import BeautifulSoup
from urllib.parse import urlparse

def extract_url_text(url):
    parsed_url = urlparse(url)

    if parsed_url.scheme not in ["http", "https"]:
        raise ValueError("Please provide a valid HTTP or HTTPS URL.")

    response = requests.get(
        url,
        timeout=15,
        headers={
            "User-Agent": "Mozilla/5.0 FakeGuard-AI"
        }
    )
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    for element in soup(["script", "style", "noscript", "svg"]):
        element.decompose()

    title = soup.title.get_text(" ", strip=True) if soup.title else ""
    page_text = soup.get_text(" ", strip=True)

    combined_text = f"{title} {page_text}".strip()

    return combined_text

print("✅ Public URL text extraction function ready.")

✅ Public URL text extraction function ready.


In [23]:
# Add a Flask endpoint that extracts text from a public URL and sends it through the ML pipeline.
@app.route("/analyze-url", methods=["POST"])
def analyze_url():
    data = request.get_json(silent=True) or {}
    url = data.get("url", "").strip()

    if not url:
        return jsonify({"error": "Please provide a job-posting URL."}), 400

    try:
        extracted_text = extract_url_text(url)

        if not extracted_text:
            return jsonify({"error": "Could not extract readable text from this URL."}), 400

        result = analyze_job_text(extracted_text)
        result["extracted_characters"] = len(extracted_text)
        result["source_url"] = url

        return jsonify(result)

    except requests.exceptions.RequestException:
        return jsonify({
            "error": "The URL could not be accessed. The website may block automated requests."
        }), 400

    except Exception as error:
        return jsonify({
            "error": f"URL analysis failed: {str(error)}"
        }), 500

print("✅ URL analysis endpoint added.")

✅ URL analysis endpoint added.


In [24]:
# Add a professional URL analysis section to the FakeGuard AI interface.
from pathlib import Path

html_path = Path("/content/fakeguard_app/templates/index.html")
html_content = html_path.read_text()

url_section = """
            <div class="url-analyzer">
                <div class="url-header">
                    <div>
                        <span class="section-label">PUBLIC JOB URL</span>
                        <h3>Analyze a Job Posting Link</h3>
                        <p>Paste a publicly accessible job-posting URL and let FakeGuard AI analyze its text.</p>
                    </div>
                </div>

                <div class="url-input-row">
                    <input
                        type="url"
                        id="jobUrl"
                        placeholder="https://example.com/job-posting"
                    >
                    <button type="button" id="urlAnalyzeButton" onclick="analyzeUrl()">
                        ANALYZE URL
                    </button>
                </div>

                <p class="url-note">
                    Works with publicly accessible pages. Login-protected or JavaScript-heavy pages may not be readable.
                </p>
            </div>
"""

marker = """
        </section>

        <section id="loadingSection"
"""

if url_section not in html_content:
    html_content = html_content.replace(
        marker,
        url_section + marker
    )

html_path.write_text(html_content)

print("✅ URL analyzer added to the FakeGuard AI interface.")

✅ URL analyzer added to the FakeGuard AI interface.


In [25]:
# Add frontend JavaScript that sends public job-posting URLs to the Flask URL analysis endpoint.
from pathlib import Path

js_path = Path("/content/fakeguard_app/static/script.js")
js_content = js_path.read_text()

url_js = """
async function analyzeUrl() {
    const urlInput = document.getElementById("jobUrl");
    const url = urlInput.value.trim();

    if (!url) {
        alert("Please enter a job-posting URL.");
        return;
    }

    showLoading();

    try {
        const response = await fetch("/analyze-url", {
            method: "POST",
            headers: {
                "Content-Type": "application/json"
            },
            body: JSON.stringify({
                url: url
            })
        });

        const data = await response.json();

        hideLoading();

        if (!response.ok) {
            alert(data.error || "URL analysis failed.");
            return;
        }

        showResult(data);

    } catch (error) {
        hideLoading();
        alert("Could not connect to the FakeGuard AI server.");
    }
}
"""

if "async function analyzeUrl()" not in js_content:
    js_content += "\n" + url_js

js_path.write_text(js_content)

print("✅ URL analysis JavaScript added.")

✅ URL analysis JavaScript added.


In [26]:
# Add professional styling for the URL analyzer section.
from pathlib import Path

css_path = Path("/content/fakeguard_app/static/style.css")
css_content = css_path.read_text()

url_css = """
.url-analyzer {
    margin-top: 24px;
    padding: 24px;
    border: 1px solid rgba(255, 255, 255, 0.08);
    border-radius: 18px;
    background: rgba(255, 255, 255, 0.03);
}

.url-header h3 {
    margin: 6px 0 8px;
}

.url-header p {
    margin: 0;
    color: #9ca9c2;
}

.url-input-row {
    display: flex;
    gap: 12px;
    margin-top: 20px;
}

.url-input-row input {
    flex: 1;
    min-width: 0;
    padding: 14px 16px;
    border: 1px solid rgba(255, 255, 255, 0.12);
    border-radius: 12px;
    background: rgba(0, 0, 0, 0.2);
    color: white;
    outline: none;
}

.url-input-row input:focus {
    border-color: #6d8cff;
}

.url-input-row button {
    padding: 14px 20px;
    border: none;
    border-radius: 12px;
    cursor: pointer;
    font-weight: 700;
}

.url-note {
    margin: 12px 0 0;
    color: #7f8ba3;
    font-size: 13px;
}

@media (max-width: 700px) {
    .url-input-row {
        flex-direction: column;
    }

    .url-input-row button {
        width: 100%;
    }
}
"""

if ".url-analyzer" not in css_content:
    css_content += "\n" + url_css

css_path.write_text(css_content)

print("✅ URL analyzer styling added.")

✅ URL analyzer styling added.


In [27]:
# Add a frontend status message so users can see which file is being analyzed.
js_path = Path("/content/fakeguard_app/static/script.js")
js_content = js_path.read_text()

status_js = """
function showFileStatus(fileName) {
    const existingStatus = document.getElementById("fileStatus");

    if (existingStatus) {
        existingStatus.textContent = "Selected: " + fileName;
        return;
    }

    const status = document.createElement("p");
    status.id = "fileStatus";
    status.className = "url-note";
    status.textContent = "Selected: " + fileName;

    const uploadGrid = document.querySelector(".upload-grid");

    if (uploadGrid) {
        uploadGrid.parentNode.insertBefore(status, uploadGrid.nextSibling);
    }
}
"""

if "function showFileStatus" not in js_content:
    js_content += "\n" + status_js

js_path.write_text(js_content)

print("✅ File status functionality added.")

✅ File status functionality added.


In [28]:
# Test the unified ML analysis pipeline with text input before restarting the web application.
test_job = """
We are hiring a junior data analyst.
Candidates should have Python, SQL, and Excel experience.
The position is full-time and offers professional development.
Apply through the company's official recruitment process.
"""

test_result = analyze_job_text(test_job)

print("Prediction:", test_result["prediction"])
print("Fraud Probability:", test_result["fraud_probability"], "%")
print("Threshold:", test_result["threshold"], "%")
print("✅ ML analysis function is working.")

Prediction: Likely Legitimate
Fraud Probability: 25.88 %
Threshold: 66.0 %
✅ ML analysis function is working.


In [29]:
# Verify that the FakeGuard AI frontend and backend files are present and ready.
from pathlib import Path

required_files = [
    Path("/content/fakeguard_app/templates/index.html"),
    Path("/content/fakeguard_app/static/style.css"),
    Path("/content/fakeguard_app/static/script.js")
]

for file_path in required_files:
    print(
        "✅" if file_path.exists() else "❌",
        file_path.name,
        f"({file_path.stat().st_size:,} bytes)" if file_path.exists() else ""
    )

print("\nFakeGuard AI application directory:")
print("/content/fakeguard_app")

✅ index.html (7,035 bytes)
✅ style.css (10,748 bytes)
✅ script.js (5,326 bytes)

FakeGuard AI application directory:
/content/fakeguard_app


In [30]:
# Start the updated FakeGuard AI application on a fresh port so all new routes and frontend changes are loaded.
from google.colab import output
import threading
import time

def run_updated_app():
    app.run(
        host="0.0.0.0",
        port=5001,
        debug=False,
        use_reloader=False
    )

updated_server_thread = threading.Thread(
    target=run_updated_app,
    daemon=True
)

updated_server_thread.start()
time.sleep(3)

print("🚀 Updated FakeGuard AI application is running.")
print("🌐 Opening the updated application...")

output.serve_kernel_port_as_window(5001)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5001
 * Running on http://172.28.0.12:5001
INFO:werkzeug:Press CTRL+C to quit


🚀 Updated FakeGuard AI application is running.
🌐 Opening the updated application...
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [31]:
# Load the improved word + character TF-IDF model into the running FakeGuard AI app.
from scipy.sparse import hstack

loaded_improved_pipeline = joblib.load(
    "/content/drive/MyDrive/fake_job_detection_model/improved_model/fake_job_detection_improved_pipeline.joblib"
)

model = loaded_improved_pipeline["model"]
word_tfidf = loaded_improved_pipeline["word_tfidf"]
char_tfidf = loaded_improved_pipeline["char_tfidf"]
threshold = loaded_improved_pipeline["threshold"]

class CombinedTextVectorizer:
    def transform(self, texts):
        word_features = word_tfidf.transform(texts)
        char_features = char_tfidf.transform(texts)
        return hstack([word_features, char_features])

tfidf = CombinedTextVectorizer()

print("✅ Improved model connected to FakeGuard AI.")
print("Model:", type(model).__name__)
print("Word features:", len(word_tfidf.vocabulary_))
print("Character features:", len(char_tfidf.vocabulary_))
print("Decision threshold:", round(threshold, 4))

✅ Improved model connected to FakeGuard AI.
Model: LogisticRegression
Word features: 20000
Character features: 15000
Decision threshold: 0.74


In [32]:
# Detect common warning signals in job-posting text to provide an understandable explanation.
import re

def detect_warning_signs(job_text):
    text = job_text.lower()
    warnings = []

    patterns = [
        (r"(registration fee|application fee|joining fee|processing fee)", "Requests an upfront registration or application fee."),
        (r"(training fee|training charges|course fee)", "Requests payment for training or a course."),
        (r"(pay.*fee|fee.*pay|payment.*required)", "Mentions a required payment or fee."),
        (r"(limited slots|limited openings|few slots|only \d+ slots)", "Uses limited-slot or scarcity language."),
        (r"(no experience required|no experience needed|without experience)", "Advertises the opportunity as requiring no experience."),
        (r"(guaranteed job|job guaranteed|100% job|guaranteed placement)", "Uses guaranteed-job or guaranteed-placement language."),
        (r"(whatsapp only|telegram only|contact only on whatsapp)", "Directs applicants to messaging platforms as the primary contact method."),
        (r"(pay.*deposit|deposit.*pay|security deposit)", "Requests a deposit or advance payment.")
    ]

    for pattern, message in patterns:
        if re.search(pattern, text):
            warnings.append(message)

    return warnings

print("✅ Warning-sign detector ready.")

✅ Warning-sign detector ready.


In [33]:
# Update the unified analyzer so every prediction includes risk level, explanation, and warning signals.
def analyze_job_text(job_text):
    if not job_text or not job_text.strip():
        return {"error": "No readable job-posting text was found."}

    warnings = detect_warning_signs(job_text)
    cleaned_text = clean_text(job_text)
    features = tfidf.transform([cleaned_text])
    probability = float(model.predict_proba(features)[0, 1])
    prediction = probability >= threshold

    if prediction:
        risk_level = "Higher Risk"
        explanation = (
            "The machine-learning model detected patterns that are associated "
            "with fraudulent job postings in its training data."
        )
    else:
        risk_level = "Lower Risk"
        explanation = (
            "The model did not detect enough patterns associated with fraudulent "
            "job postings to cross the current decision threshold."
        )

    return {
        "prediction": "Potentially Fraudulent" if prediction else "Likely Legitimate",
        "risk_level": risk_level,
        "fraud_probability": round(probability * 100, 2),
        "threshold": round(threshold * 100, 2),
        "warning_signs": warnings,
        "warning_count": len(warnings),
        "explanation": explanation
    }

print("✅ FakeGuard AI now returns explainable analysis.")

✅ FakeGuard AI now returns explainable analysis.


In [34]:
# Add a professional explanation panel to the existing FakeGuard AI result section.
from IPython.display import display, HTML

explanation_css = """
<style>
#riskExplanation {
    margin-top: 24px;
    padding: 22px;
    border-radius: 16px;
    background: rgba(255,255,255,0.04);
    border: 1px solid rgba(255,255,255,0.10);
}

#riskExplanation h3 {
    margin: 0 0 10px 0;
}

#riskExplanation p {
    line-height: 1.6;
}

.warning-list {
    margin-top: 12px;
    padding-left: 20px;
}

.warning-list li {
    margin-bottom: 8px;
}
</style>
"""

explanation_js = """
<script>
function renderRiskExplanation(data) {
    const resultSection = document.getElementById("resultSection");

    if (!resultSection) {
        return;
    }

    let panel = document.getElementById("riskExplanation");

    if (!panel) {
        panel = document.createElement("div");
        panel.id = "riskExplanation";
        resultSection.appendChild(panel);
    }

    const warnings = data.warning_signs || [];

    let warningHTML = "";

    if (warnings.length > 0) {
        warningHTML = `
            <h3>⚠️ Warning Signals Detected</h3>
            <ul class="warning-list">
                ${warnings.map(item => `<li>${item}</li>`).join("")}
            </ul>
        `;
    } else {
        warningHTML = `
            <h3>🔎 Warning Signals</h3>
            <p>No predefined warning signals were detected in the submitted text.</p>
        `;
    }

    panel.innerHTML = `
        <h3>💡 Why did FakeGuard give this result?</h3>
        <p>${data.explanation || "The model analyzed the submitted job-posting text."}</p>
        <p><strong>Risk level:</strong> ${data.risk_level || "Not available"}</p>
        ${warningHTML}
        <p style="margin-top:18px;">
            <strong>Important:</strong> These signals are indicators, not proof that a
            job posting is fraudulent. Always verify the employer through its official
            website and never pay money to obtain a job.
        </p>
    `;
}

const originalShowResult = showResult;

showResult = function(data) {
    originalShowResult(data);
    renderRiskExplanation(data);
};
</script>
"""

display(HTML(explanation_css + explanation_js))

print("✅ Explanation panel added to the FakeGuard AI interface.")

✅ Explanation panel added to the FakeGuard AI interface.


In [35]:
# Test the complete improved analyzer using the suspicious job-posting example.
test_job = """
No experience, limited slots, AI engineer, registration fee 2000.
"""

test_result = analyze_job_text(test_job)

print("Prediction:", test_result["prediction"])
print("Risk level:", test_result["risk_level"])
print("Fraud probability:", test_result["fraud_probability"], "%")
print("Decision threshold:", test_result["threshold"], "%")
print("\nExplanation:")
print(test_result["explanation"])
print("\nWarning signals:")

for warning in test_result["warning_signs"]:
    print("-", warning)

Prediction: Likely Legitimate
Risk level: Lower Risk
Fraud probability: 48.97 %
Decision threshold: 74.0 %

Explanation:
The model did not detect enough patterns associated with fraudulent job postings to cross the current decision threshold.

Warning signals:
- Requests an upfront registration or application fee.
- Uses limited-slot or scarcity language.


In [36]:
# Replace the existing prediction route with the improved analyzer without registering a new Flask route.
from flask import request, jsonify

def improved_predict():
    data = request.get_json(silent=True) or {}
    job_text = data.get("text", "").strip()

    if not job_text:
        return jsonify({
            "error": "Please provide job posting text."
        }), 400

    result = analyze_job_text(job_text)

    if "error" in result:
        return jsonify(result), 400

    return jsonify(result)

app.view_functions["predict"] = improved_predict

print("✅ Existing /predict route now uses the improved analyzer.")

✅ Existing /predict route now uses the improved analyzer.


In [37]:
# Replace the existing file-analysis route so PDF and image uploads use the improved analyzer.
def improved_analyze_file():
    if "file" not in request.files:
        return jsonify({"error": "No file was uploaded."}), 400

    uploaded_file = request.files["file"]

    if not uploaded_file.filename:
        return jsonify({"error": "Please select a file."}), 400

    filename = secure_filename(uploaded_file.filename)
    extension = filename.lower().split(".")[-1]

    try:
        if extension == "pdf":
            extracted_text = extract_pdf_text(uploaded_file)
        elif extension in ["png", "jpg", "jpeg", "webp"]:
            extracted_text = extract_image_text(uploaded_file)
        else:
            return jsonify({
                "error": "Supported formats are PDF, PNG, JPG, JPEG, and WEBP."
            }), 400

        if not extracted_text:
            return jsonify({
                "error": "Could not extract readable text from this file."
            }), 400

        result = analyze_job_text(extracted_text)
        result["extracted_characters"] = len(extracted_text)

        return jsonify(result)

    except Exception as error:
        return jsonify({
            "error": f"File analysis failed: {str(error)}"
        }), 500

app.view_functions["analyze_file"] = improved_analyze_file

print("✅ Existing file-analysis route now uses the improved analyzer.")

✅ Existing file-analysis route now uses the improved analyzer.


In [38]:
# Replace the existing URL-analysis route so URL analysis uses the improved analyzer.
def improved_analyze_url():
    data = request.get_json(silent=True) or {}
    url = data.get("url", "").strip()

    if not url:
        return jsonify({
            "error": "Please provide a job-posting URL."
        }), 400

    try:
        extracted_text = extract_url_text(url)

        if not extracted_text:
            return jsonify({
                "error": "Could not extract readable text from this URL."
            }), 400

        result = analyze_job_text(extracted_text)
        result["extracted_characters"] = len(extracted_text)
        result["source_url"] = url

        return jsonify(result)

    except requests.exceptions.RequestException:
        return jsonify({
            "error": "The URL could not be accessed. The website may block automated requests."
        }), 400

    except Exception as error:
        return jsonify({
            "error": f"URL analysis failed: {str(error)}"
        }), 500

app.view_functions["analyze_url"] = improved_analyze_url

print("✅ Existing URL-analysis route now uses the improved analyzer.")

✅ Existing URL-analysis route now uses the improved analyzer.


In [39]:
# Verify that the existing Flask endpoints are connected to the updated functions.
for rule in app.url_map.iter_rules():
    print(f"{rule.methods}  {rule}")

{'OPTIONS', 'HEAD', 'GET'}  /static/<path:filename>
{'OPTIONS', 'HEAD', 'GET'}  /health
{'OPTIONS', 'POST'}  /predict
{'OPTIONS', 'HEAD', 'GET'}  /
{'OPTIONS', 'POST'}  /analyze-file
{'OPTIONS', 'POST'}  /analyze-url


In [40]:
# Test the improved prediction endpoint directly before refreshing the web interface.
test_client = app.test_client()

test_response = test_client.post(
    "/predict",
    json={
        "text": """
        No experience, limited slots, AI engineer, registration fee 2000.
        """
    }
)

print("HTTP status:", test_response.status_code)
print("Response:")
print(test_response.get_json())

HTTP status: 200
Response:
{'explanation': 'The model did not detect enough patterns associated with fraudulent job postings to cross the current decision threshold.', 'fraud_probability': 48.97, 'prediction': 'Likely Legitimate', 'risk_level': 'Lower Risk', 'threshold': 74.0, 'warning_count': 2, 'warning_signs': ['Requests an upfront registration or application fee.', 'Uses limited-slot or scarcity language.']}


In [41]:
# Make the result wording clearer so a lower model score is not mistaken for proof that a job is safe.
def analyze_job_text(job_text):
    if not job_text or not job_text.strip():
        return {"error": "No readable job-posting text was found."}

    warnings = detect_warning_signs(job_text)
    cleaned_text = clean_text(job_text)
    features = tfidf.transform([cleaned_text])
    probability = float(model.predict_proba(features)[0, 1])
    prediction = probability >= threshold

    if prediction:
        risk_level = "Model Flagged"
        explanation = (
            "The model score crossed the decision threshold and the posting "
            "contains patterns associated with fraudulent postings in the training data."
        )
    elif warnings:
        risk_level = "Review Carefully"
        explanation = (
            "The model score is below the fraud decision threshold, but the "
            "submitted posting contains warning signals that deserve manual verification."
        )
    else:
        risk_level = "No Model Flag"
        explanation = (
            "The model score is below the fraud decision threshold and no predefined "
            "warning signals were detected in the submitted text."
        )

    return {
        "prediction": "Potentially Fraudulent" if prediction else "Likely Legitimate",
        "risk_level": risk_level,
        "fraud_probability": round(probability * 100, 2),
        "threshold": round(threshold * 100, 2),
        "warning_signs": warnings,
        "warning_count": len(warnings),
        "explanation": explanation
    }

print("✅ Result wording improved.")

✅ Result wording improved.


In [42]:
# Test the updated explanation using the same suspicious example.
test_job = """
No experience, limited slots, AI engineer, registration fee 2000.
"""

result = analyze_job_text(test_job)

print("Prediction:", result["prediction"])
print("Risk level:", result["risk_level"])
print("Fraud probability:", result["fraud_probability"], "%")
print("Threshold:", result["threshold"], "%")
print("Warning count:", result["warning_count"])
print("\nExplanation:", result["explanation"])

Prediction: Likely Legitimate
Risk level: Review Carefully
Fraud probability: 48.97 %
Threshold: 74.0 %
Warning count: 2

Explanation: The model score is below the fraud decision threshold, but the submitted posting contains warning signals that deserve manual verification.


In [43]:
# Test the analyzer with a normal-looking job posting to verify the other result path.
normal_job = """
We are hiring a Data Analyst with experience in Python, SQL, Excel, and data visualization.
The selected candidate will work with our analytics team and report to the Data Science Manager.
Please apply through our official company careers website.
"""

normal_result = analyze_job_text(normal_job)

print("Prediction:", normal_result["prediction"])
print("Risk level:", normal_result["risk_level"])
print("Fraud probability:", normal_result["fraud_probability"], "%")
print("Threshold:", normal_result["threshold"], "%")
print("Warning count:", normal_result["warning_count"])
print("\nExplanation:", normal_result["explanation"])

Prediction: Likely Legitimate
Risk level: No Model Flag
Fraud probability: 16.96 %
Threshold: 74.0 %
Warning count: 0

Explanation: The model score is below the fraud decision threshold and no predefined warning signals were detected in the submitted text.


In [44]:
# Verify that the live Flask prediction endpoint returns the new risk wording.
response = test_client.post(
    "/predict",
    json={
        "text": """
        No experience, limited slots, AI engineer, registration fee 2000.
        """
    }
)

print("HTTP status:", response.status_code)
print(response.get_json())

HTTP status: 200
{'explanation': 'The model score is below the fraud decision threshold, but the submitted posting contains warning signals that deserve manual verification.', 'fraud_probability': 48.97, 'prediction': 'Likely Legitimate', 'risk_level': 'Review Carefully', 'threshold': 74.0, 'warning_count': 2, 'warning_signs': ['Requests an upfront registration or application fee.', 'Uses limited-slot or scarcity language.']}


In [45]:
# Verify that the FakeGuard AI application has all required endpoints registered.
required_routes = ["/", "/predict", "/analyze-file", "/analyze-url"]

registered_routes = {
    rule.rule
    for rule in app.url_map.iter_rules()
}

for route in required_routes:
    status = "✅" if route in registered_routes else "❌"
    print(f"{status} {route}")

print("\nFakeGuard AI backend verification complete.")

✅ /
✅ /predict
✅ /analyze-file
✅ /analyze-url

FakeGuard AI backend verification complete.


In [46]:
# Add polished styling for the risk explanation and warning-sign sections.
explanation_style = """
<style>
#riskExplanation {
    margin-top: 24px;
    padding: 24px;
    border-radius: 18px;
    background: rgba(255,255,255,0.045);
    border: 1px solid rgba(255,255,255,0.10);
}

#riskExplanation .explanation-title {
    font-size: 18px;
    font-weight: 700;
    margin-bottom: 10px;
}

#riskExplanation .risk-status {
    display: inline-block;
    margin: 8px 0 16px 0;
    padding: 7px 12px;
    border-radius: 999px;
    font-weight: 700;
    font-size: 13px;
}

#riskExplanation .explanation-text {
    line-height: 1.65;
    margin-bottom: 18px;
}

#riskExplanation .warning-title {
    font-size: 16px;
    font-weight: 700;
    margin-bottom: 10px;
}

#riskExplanation ul {
    margin: 8px 0 18px 20px;
    padding: 0;
}

#riskExplanation li {
    margin-bottom: 9px;
    line-height: 1.5;
}

#riskExplanation .verification-box {
    margin-top: 18px;
    padding: 15px;
    border-radius: 12px;
    background: rgba(255,255,255,0.035);
    line-height: 1.6;
}
</style>
"""

from IPython.display import display, HTML
display(HTML(explanation_style))

print("✅ Explanation styling added.")

✅ Explanation styling added.


In [47]:
# Create the frontend renderer that displays model reasoning, risk status, and warning signals.
explanation_renderer = """
<script>
function renderRiskExplanation(data) {
    const resultSection = document.getElementById("resultSection");

    if (!resultSection) {
        return;
    }

    let panel = document.getElementById("riskExplanation");

    if (!panel) {
        panel = document.createElement("div");
        panel.id = "riskExplanation";
        resultSection.appendChild(panel);
    }

    const warnings = data.warning_signs || [];
    const riskLevel = data.risk_level || "Not available";

    let warningHTML = "";

    if (warnings.length > 0) {
        warningHTML = `
            <div class="warning-title">⚠️ Warning Signals Detected</div>
            <ul>
                ${warnings.map(warning => `<li>${warning}</li>`).join("")}
            </ul>
        `;
    } else {
        warningHTML = `
            <div class="warning-title">🔎 Warning Signals</div>
            <p>No predefined warning signals were detected in the submitted text.</p>
        `;
    }

    panel.innerHTML = `
        <div class="explanation-title">💡 Why did FakeGuard give this result?</div>

        <div class="risk-status">${riskLevel}</div>

        <div class="explanation-text">
            ${data.explanation || "The model analyzed the submitted job posting."}
        </div>

        ${warningHTML}

        <div class="verification-box">
            <strong>🛡️ Before applying:</strong><br>
            Verify the employer through its official website, check the company's
            contact details, and be cautious if the opportunity requires upfront
            payment or other fees.
        </div>
    `;
}
</script>
"""

display(HTML(explanation_renderer))

print("✅ Explanation renderer created.")

✅ Explanation renderer created.


In [48]:
# Connect the explanation panel to the existing JavaScript result display without referencing JavaScript from Python.
from IPython.display import display, HTML

display(HTML("""
<script>
if (typeof window.originalFakeGuardShowResult === "undefined" &&
    typeof window.showResult === "function") {

    window.originalFakeGuardShowResult = window.showResult;

    window.showResult = function(data) {
        window.originalFakeGuardShowResult(data);

        if (typeof window.renderRiskExplanation === "function") {
            window.renderRiskExplanation(data);
        }
    };

    console.log("✅ FakeGuard explanation connected.");
} else {
    console.log("ℹ️ FakeGuard explanation connection already exists.");
}
</script>
"""))

print("✅ Explanation connection script added.")

✅ Explanation connection script added.


In [49]:
# Check which backend endpoints are currently referenced by the FakeGuard frontend.
script_path = STATIC_DIR / "script.js"
script_text = script_path.read_text()

print("Frontend API connections:")

for endpoint in ["/predict", "/analyze-file", "/analyze-file-v2", "/analyze-url", "/analyze-url-v2"]:
    if endpoint in script_text:
        print("✅", endpoint)

Frontend API connections:
✅ /predict
✅ /analyze-file
✅ /analyze-url


In [50]:
# Connect PDF and image uploads to the improved file-analysis endpoint.
script_text = script_path.read_text()

script_text = script_text.replace(
    'fetch("/analyze-file", {',
    'fetch("/analyze-file-v2", {'
)

script_path.write_text(script_text)

print("✅ PDF/image analysis connected to the improved endpoint.")

✅ PDF/image analysis connected to the improved endpoint.


In [51]:
# Connect public job URL analysis to the improved URL-analysis endpoint.
script_text = script_path.read_text()

script_text = script_text.replace(
    'fetch("/analyze-url", {',
    'fetch("/analyze-url-v2", {'
)

script_path.write_text(script_text)

print("✅ URL analysis connected to the improved endpoint.")

✅ URL analysis connected to the improved endpoint.


In [52]:
# Verify that the frontend JavaScript contains the expected API connections.
script_text = script_path.read_text()

checks = {
    "Text analysis": "/predict",
    "PDF/image analysis": "/analyze-file-v2",
    "URL analysis": "/analyze-url-v2"
}

for name, endpoint in checks.items():
    if endpoint in script_text:
        print(f"✅ {name}: {endpoint}")
    else:
        print(f"❌ {name}: {endpoint} not found")

print("\n✅ Frontend connection check complete.")

✅ Text analysis: /predict
✅ PDF/image analysis: /analyze-file-v2
✅ URL analysis: /analyze-url-v2

✅ Frontend connection check complete.


In [53]:
# Check the existing Flask routes before making any further backend changes.
print("Current FakeGuard AI routes:\n")

for rule in app.url_map.iter_rules():
    methods = ", ".join(sorted(rule.methods - {"HEAD", "OPTIONS"}))
    print(f"{methods:15} {rule.rule}")

Current FakeGuard AI routes:

GET             /static/<path:filename>
GET             /health
POST            /predict
GET             /
POST            /analyze-file
POST            /analyze-url


In [54]:
# Connect the improved file analyzer to the existing file route without registering a new Flask route.
if "analyze_file" in app.view_functions:
    app.view_functions["analyze_file"] = improved_analyze_file
    print("✅ /analyze-file now uses the improved analyzer.")
else:
    print("⚠️ /analyze-file was not found.")

✅ /analyze-file now uses the improved analyzer.


In [55]:
# Connect the improved URL analyzer to the existing URL route without registering a new Flask route.
if "analyze_url" in app.view_functions:
    app.view_functions["analyze_url"] = improved_analyze_url
    print("✅ /analyze-url now uses the improved analyzer.")
else:
    print("⚠️ /analyze-url was not found.")

✅ /analyze-url now uses the improved analyzer.


In [56]:
# Test the live text endpoint with a posting containing two warning signals.
text_test = test_client.post(
    "/predict",
    json={
        "text": """
        No experience, limited slots, AI engineer, registration fee 2000.
        """
    }
)

print("HTTP status:", text_test.status_code)
print(text_test.get_json())

HTTP status: 200
{'explanation': 'The model score is below the fraud decision threshold, but the submitted posting contains warning signals that deserve manual verification.', 'fraud_probability': 48.97, 'prediction': 'Likely Legitimate', 'risk_level': 'Review Carefully', 'threshold': 74.0, 'warning_count': 2, 'warning_signs': ['Requests an upfront registration or application fee.', 'Uses limited-slot or scarcity language.']}


In [57]:
# Verify that the FakeGuard AI homepage is still available after the backend updates.
home_test = test_client.get("/")

print("HTTP status:", home_test.status_code)
print("Homepage available:", home_test.status_code == 200)

if home_test.status_code == 200:
    print("✅ FakeGuard AI homepage is working.")
else:
    print("❌ Homepage returned an unexpected status.")

HTTP status: 200
Homepage available: True
✅ FakeGuard AI homepage is working.


In [58]:
# Check the active FakeGuard AI routes

for rule in app.url_map.iter_rules():
    print(rule.rule, "->", rule.endpoint)

/static/<path:filename> -> static
/health -> health
/predict -> predict
/ -> home
/analyze-file -> analyze_file
/analyze-url -> analyze_url


In [59]:
# Connect the improved file analyzer to the existing route

app.view_functions["analyze_file"] = improved_analyze_file

print("✅ /analyze-file connected to improved analyzer")

✅ /analyze-file connected to improved analyzer


In [60]:
# Connect the improved URL analyzer to the existing route

app.view_functions["analyze_url"] = improved_analyze_url

print("✅ /analyze-url connected to improved analyzer")

✅ /analyze-url connected to improved analyzer


In [61]:
# Restore frontend endpoints to the existing Flask routes

script_path = "/content/fakeguard_app/static/script.js"

with open(script_path, "r", encoding="utf-8") as f:
    js = f.read()

js = js.replace("/analyze-file-v2", "/analyze-file")
js = js.replace("/analyze-url-v2", "/analyze-url")

with open(script_path, "w", encoding="utf-8") as f:
    f.write(js)

print("✅ Frontend endpoints fixed")

✅ Frontend endpoints fixed


In [62]:
# Verify the frontend is using the correct endpoints

with open(script_path, "r", encoding="utf-8") as f:
    js = f.read()

print("analyze-file present:", "/analyze-file" in js)
print("analyze-url present:", "/analyze-url" in js)
print("old file endpoint present:", "/analyze-file-v2" in js)
print("old URL endpoint present:", "/analyze-url-v2" in js)

analyze-file present: True
analyze-url present: True
old file endpoint present: False
old URL endpoint present: False


In [63]:
# Show the FakeGuard AI web app link again

from google.colab import output

print("🚀 FakeGuard AI is running on port 5000")
print("👇 Click the link below to open the website")

output.serve_kernel_port_as_window(5000)

🚀 FakeGuard AI is running on port 5000
👇 Click the link below to open the website
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>